# Neptune AgentCore Gateway - Local Testing

Tests the deployed AgentCore Gateway with its Amazon Neptune tools.

**Tools available (9):**
- `discover_named_graphs` - List all named graphs
- `get_ontology_from_neptune` - Retrieve full ontology by `ontology_id` (UUID)
- `persist_to_neptune` - Write RDF n-quad data
- `delete_graph` - Drop all triples in a named graph by `ontology_id`
- `execute_sparql_query` - Execute generic SPARQL queries
- `get_graph_summary` - Summary statistics
- `get_graph_stats` - Class distribution statistics
- `get_graph_classes` - List all classes
- `get_graph_properties` - List all properties

## Step 1: Install dependencies

In [ ]:
# Prereq (uncomment if not already installed):
# !pip install boto3 mcp-proxy-for-aws strands-agents --quiet

In [ ]:
# Configure boto3 session with AWS profile
import boto3
from botocore.config import Config
from dotenv import load_dotenv
import os

config = Config(
    retries={
        'max_attempts': 10,
        'mode': 'standard'
    },
    signature_version='v4'
)

# Load environment variables from .env file (falls back to os.environ if not found)
load_dotenv(dotenv_path='.env', override=False)

# Create session with specific AWS profile
session = boto3.Session(profile_name=os.environ.get('AWS_PROFILE', 'XXX'))
region = session.region_name or 'us-east-1'

# Create STS client to verify credentials
sts = session.client(
    service_name='sts',
    region_name=region,
    config=config
)

# Verify credentials work
try:
    identity = sts.get_caller_identity()
    print(f"✓ AWS Credentials Verified:")
    print(f"  Profile: {os.environ.get('AWS_PROFILE', 'XXX')}")
    print(f"  Account: ...{identity['Account'][-4:]}")
    print(f"  User/Role: {identity['Arn'].split('/')[-1]}")
    print(f"  Region: {region}")
except Exception as e:
    print(f"✗ Failed to verify AWS credentials: {str(e)}")
    raise

## Step 2: Load Gateway configuration from SSM

In [ ]:
import boto3
import os


PROJECT_NAME = os.environ.get('PROJECT_NAME', 'XXX')



ssm = session.client(
    service_name='ssm',
    region_name=region,
    config=config
)

def get_ssm_param(name):
    resp = ssm.get_parameter(Name=name)
    return resp['Parameter']['Value']

GATEWAY_URL = get_ssm_param(f'/{PROJECT_NAME}/neptune-gateway/url')
GATEWAY_ID  = get_ssm_param(f'/{PROJECT_NAME}/neptune-gateway/id')

print(f'Gateway URL : {GATEWAY_URL[:20]}...')
print(f'Gateway ID  : {GATEWAY_ID}')

## Step 3: List available tools

Verify the Gateway is reachable and all 8 Neptune tools are registered.

In [ ]:
from strands.tools.mcp import MCPClient
from mcp_proxy_for_aws.client import aws_iam_streamablehttp_client

# Pass credentials from the already-authenticated huthmac+demo session so that
# aws_iam_streamablehttp_client uses SigV4 IAM signing with those credentials,
# rather than creating a new default boto3 session that picks up expired SSO tokens.
mcp_client = MCPClient(
    lambda: aws_iam_streamablehttp_client(
        endpoint=GATEWAY_URL,
        aws_region=region,
        aws_service='bedrock-agentcore',
        credentials=session.get_credentials(),
    )
)

with mcp_client:
    tools = mcp_client.list_tools_sync()

print(f'Found {len(tools)} tools:')
for t in tools:
    print(f'  - {t.tool_name}')


## Step 4: Individual tool invocations

In [ ]:
import uuid, json

# Cache: short tool name → full MCP name registered by the Gateway
# e.g. 'discover_named_graphs' → 'discover-named-graphs___discover_named_graphs'
_tool_name_cache: dict = {}

def call_tool(short_name: str, arguments: dict | None = None):
    """
    Invoke a Neptune Gateway tool by its short name (e.g. 'discover_named_graphs').

    On the first call the helper connects, lists all tools, and builds a mapping
    from the suffix after '___' to the full Gateway-prefixed name that the MCP
    protocol requires.  Subsequent calls reuse the cached mapping.
    """
    global _tool_name_cache

    with mcp_client:
        # Populate cache once per kernel session
        if not _tool_name_cache:
            tools = mcp_client.list_tools_sync()
            for t in tools:
                full  = t.mcp_tool.name                       # 'discover-named-graphs___discover_named_graphs'
                short = full.split('___')[-1].replace('-', '_') # 'discover_named_graphs'
                _tool_name_cache[short] = full
            print(f'Tool name cache built: {list(_tool_name_cache.keys())}')

        full_name = _tool_name_cache.get(short_name)
        if full_name is None:
            raise ValueError(f"Unknown tool '{short_name}'. Available: {list(_tool_name_cache.keys())}")

        result = mcp_client.call_tool_sync(
            tool_use_id=str(uuid.uuid4()),
            name=full_name,
            arguments=arguments or {},
        )

    for item in result['content']:
        raw = item.get('text') or item.get('json')
        if raw is None:
            raw = str(item)
        if isinstance(raw, (dict, list)):
            print(json.dumps(raw, indent=2))
        else:
            try:
                print(json.dumps(json.loads(raw), indent=2))
            except (json.JSONDecodeError, TypeError):
                print(str(raw))


In [ ]:
# ── discover_named_graphs ────────────────────────────────────────────────────
# Lists all named graphs (ontologies) stored in the Neptune database.
# No input parameters required.

call_tool('discover_named_graphs')


In [ ]:
# ── execute_sparql_query ─────────────────────────────────────────────────────
# Executes a generic SPARQL query (SELECT or UPDATE) against Neptune.

sparql_query = 'SELECT ?s ?p ?o WHERE { ?s ?p ?o } LIMIT 10'
query_type   = 'SELECT'   # 'SELECT' or 'UPDATE'

call_tool('execute_sparql_query', {
    'sparql_query': sparql_query,
    'query_type':   query_type,
})


In [ ]:
# ── Resolve a real ontology_id to test the graph-scoped tools ────────────────
# Prefer a graph that actually exists in Neptune: take the first id from
# discover_named_graphs; otherwise fall back to the latest completed
# vkg-ontology-* layer in the metadata table (its id is the Neptune graph name).
import json as _json


def _first_named_graph_id() -> str:
    """Return the first ontology_id from discover_named_graphs, or '' if none."""
    with mcp_client:
        if not _tool_name_cache:
            for t in mcp_client.list_tools_sync():
                full = t.mcp_tool.name
                _tool_name_cache[full.split('___')[-1].replace('-', '_')] = full
        res = mcp_client.call_tool_sync(
            tool_use_id=str(uuid.uuid4()),
            name=_tool_name_cache['discover_named_graphs'], arguments={})
    for item in res['content']:
        raw = item.get('text') or item.get('json')
        try:
            body = _json.loads(raw) if isinstance(raw, str) else raw
            inner = _json.loads(body['body']) if isinstance(body.get('body'), str) else body
            graphs = inner.get('named_graphs') or []
            if graphs:
                # named graph entries may be IRIs ending in the ontology id
                g = graphs[0]
                gid = g if isinstance(g, str) else (g.get('graph') or g.get('id') or '')
                return gid.rstrip('/#').rsplit('/', 1)[-1]
        except Exception:
            continue
    return ''


def _latest_completed_vkg_id() -> str:
    """Newest completed vkg-ontology-* layer id from the metadata table."""
    tbl = session.resource('dynamodb', region_name=region).Table(
        os.environ.get('ONTOLOGY_METADATA_TABLE', f'{PROJECT_NAME}-metadata'))
    items = tbl.scan(
        FilterExpression='contains(id, :p) AND #s = :c',
        ExpressionAttributeNames={'#s': 'status'},
        ExpressionAttributeValues={':p': 'vkg-ontology', ':c': 'completed'},
    ).get('Items', [])
    items.sort(key=lambda it: it.get('updatedAt') or it.get('createdAt') or '', reverse=True)
    return items[0]['id'] if items else ''


ontology_id = _first_named_graph_id() or _latest_completed_vkg_id()
if not ontology_id:
    print("⚠ No ontology graph found in Neptune and no completed VKG layer — "
          "the graph-scoped tool cells below will return an empty/no-graph result.")
else:
    print(f"✓ Using ontology_id: {ontology_id}")

# get_graph_summary — class count, property count, triple count for the graph.
call_tool('get_graph_summary', {
    'ontology_id': ontology_id,
})

In [ ]:
# ── get_graph_stats ──────────────────────────────────────────────────────────
# Returns the top-20 classes by instance count for an ontology graph.


call_tool('get_graph_stats', {
    'ontology_id': ontology_id,
})


In [ ]:
# ── get_graph_classes ────────────────────────────────────────────────────────
# Returns all OWL classes (URIs, labels, comments) in an ontology graph.

call_tool('get_graph_classes', {
    'ontology_id': ontology_id,
})


In [ ]:
# ── get_graph_properties ─────────────────────────────────────────────────────
# Returns all RDF properties (URIs, labels, comments) in an ontology graph.

call_tool('get_graph_properties', {
    'ontology_id': ontology_id,
})


In [ ]:
# ── get_ontology_from_neptune ────────────────────────────────────────────────
# Retrieves the full ontology (classes, properties, mappings) for an ontology_id.
# Reuses the ontology_id resolved above (discover_named_graphs → metadata fallback).

call_tool('get_ontology_from_neptune', {
    'ontology_id': ontology_id,
})

In [ ]:
# # ── persist_to_neptune ───────────────────────────────────────────────────────
# # Writes RDF n-quad data to Neptune via SPARQL INSERT DATA.
# # Format: <subject> <predicate> <object> <named-graph> .

# nquad_data = """<http://example.com/test/subject1> <http://www.w3.org/1999/02/22-rdf-syntax-ns#type> <http://www.w3.org/2002/07/owl#Class> <http://example.com/test/graph> .
# <http://example.com/test/subject1> <http://www.w3.org/2000/01/rdf-schema#label> "TestClass" <http://example.com/test/graph> ."""

# call_tool('persist_to_neptune', {
#     'nquad_data': nquad_data,
# })
